# [NEXON Open API] 마비노기 경매장 거래 내역 수집

**데이터 출처**
  * Data based on NEXON Open API (https://openapi.nexon.com)

**수집 내용**
| 데이터 | 엔드포인트 | 수집 주기 |
| :--- | :--- | :--- |
| 경매장 거래 내역 (최근 1시간) | `/mabinogi/v1/auction/history` | 30분마다 (매시 00분, 30분) |

* 수집한 원본 데이터는 BigQuery에만 저장하며, 외부에 공개하지 않습니다.

In [ ]:
# 환경 설정
import os
import json
import time
from datetime import datetime
from zoneinfo import ZoneInfo

import requests
import pandas as pd
from dotenv import load_dotenv

from google.cloud import bigquery

In [ ]:
# .env에서 API 키 불러오기
load_dotenv("../.env", override=True)

GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID")
MABINOGI_KEY = os.getenv("NEXON_MABINOGI_API_KEY")

print("API 키 불러오기: ", "성공" if MABINOGI_KEY else "실패")

# api 키는 헤더에 넣어서 보내기
headers = {
    "x-nxopen-api-key": MABINOGI_KEY
}

# 수집 시각 (한국 시간)
collected_at = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
# 마비노기 경매장 거래 내역 api (최근 1시간 데이터 제공)
API_URL = "https://open.api.nexon.com/mabinogi/v1/auction/history"

auction_history = []
failed_count = 0
cursor = ""

# 한 번에 500건씩 오기 때문에 다음 페이지가 없을 때까지 반복 (최대 40페이지)
for page in range(40):
    params = {
        "cursor": cursor
    }

    try:
        response = requests.get(API_URL, params=params, headers=headers, timeout=30)

        # HTTP 요청에 문제가 있으면 오류 발생하고 중단
        response.raise_for_status()

        # 가져온 api를 json 형태로 변경
        data = response.json()

        # 거래 내역마다 수집 시각을 붙여서 저장
        for item in data["auction_history"]:
            item["item_option"] = json.dumps(item.get("item_option"), ensure_ascii=False)
            item["collected_at"] = collected_at
            auction_history.append(item)

        # 다음 페이지 cursor가 없으면 반복 종료
        cursor = data.get("next_cursor")
        if not cursor:
            break

        time.sleep(1)

    except Exception as e:
        print(f"{page + 1}페이지 요청 실패: {e}")
        failed_count += 1
        break

auction_history_df = pd.DataFrame(auction_history)

print("수집 완료된 데이터 개수:", len(auction_history))
print("수집 실패한 페이지 개수:", failed_count)

In [ ]:
# BigQuery 연결
client = bigquery.Client(project=GCP_PROJECT_ID)

def upload_to_bigquery(df, table_id):
    # 데이터가 없으면 건너뛰기
    if df is None or df.empty:
        print(f"{table_id}: 업로드할 데이터 없음")
        return

    df = df.copy()

    # 글자 컬럼은 문자열 타입으로 고정 (실행할 때마다 타입이 바뀌지 않도록)
    for col in df.select_dtypes(include=["object", "str"]).columns:
        df[col] = df[col].astype("string")

    # 기존 테이블 아래에 이어 붙이기 (테이블이 없으면 자동으로 생성)
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")

    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()   # 업로드가 끝날 때까지 기다리기

    print(f"{table_id}: {len(df):,}행 업로드 완료")


upload_to_bigquery(auction_history_df, f"{GCP_PROJECT_ID}.mabinogi.auction_history")